# Word Frequency Analysis of Moby Dick

This notebook walks through the fundamentals of **Natural Language Processing (NLP)** using Herman Melville's *Moby Dick* as the corpus.

By the end, you will have covered:
- Scraping plain text from Project Gutenberg
- Cleaning and normalising raw text
- Tokenisation — splitting text into individual words
- Stopword removal — filtering out words with no analytical value
- Frequency distribution — counting and ranking every unique word
- Visualising the top-N words as a bar chart
- Verifying **Zipf's Law** on a real corpus via a log-log plot

---

### What is Zipf's Law?

In any large body of natural language text, the frequency of a word is inversely proportional to its rank in the frequency table.

Put plainly: the most common word appears roughly **twice** as often as the second most common, **three times** as often as the third, and so on.

This holds across virtually every language and every text — it is one of the most striking empirical regularities in linguistics.

## 0. Setup

Install dependencies once (comment out after first run), then download the NLTK data packages this notebook depends on.

In [ ]:
# Uncomment and run once if you have not installed dependencies yet
# !pip install requests beautifulsoup4 nltk matplotlib numpy

import nltk

# punkt      — sentence and word tokeniser models
# stopwords  — lists of common words per language
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

print('NLTK data ready.')

## 1. Fetch the text

Project Gutenberg hosts the full plain-text of Moby Dick at a stable URL. We fetch it over HTTP — no authentication, no API key.

The raw response includes a Gutenberg **header** and **footer** (licensing information, metadata) that are not part of the novel. We strip those out before any analysis.

In [ ]:
import requests

GUTENBERG_URL = 'https://www.gutenberg.org/files/2701/2701-0.txt'

print(f'Fetching: {GUTENBERG_URL}')
response = requests.get(GUTENBERG_URL, timeout=30)
response.raise_for_status()  # Raises an error for any non-200 HTTP status

raw_text = response.text
print(f'Downloaded {len(raw_text):,} characters.')

In [ ]:
# Gutenberg wraps every book with a standard header and footer.
# The novel's content sits between these two sentinel strings.

START_MARKER = '*** START OF THE PROJECT GUTENBERG EBOOK MOBY DICK; OR THE WHALE ***'
END_MARKER   = '*** END OF THE PROJECT GUTENBERG EBOOK MOBY DICK; OR THE WHALE ***'

start_idx = raw_text.find(START_MARKER)
end_idx   = raw_text.find(END_MARKER)

if start_idx == -1 or end_idx == -1:
    raise ValueError('Could not find Gutenberg markers — the file format may have changed.')

# Slice out just the novel text, skipping past the marker line itself
novel_text = raw_text[start_idx + len(START_MARKER) : end_idx].strip()

print(f'Novel text extracted: {len(novel_text):,} characters.')
print()
print('--- First 500 characters ---')
print(novel_text[:500])

## 2. Tokenise

**Tokenisation** is the process of splitting a string of text into a list of individual tokens (usually words and punctuation marks).

NLTK's `word_tokenize` handles contractions (`can't` → `ca`, `n't`), punctuation splitting, and edge cases that a naive `str.split()` would miss.

We also **lowercase** everything so that `Whale`, `WHALE`, and `whale` are all counted as the same word.

In [ ]:
from nltk.tokenize import word_tokenize

# Lowercase before tokenising so casing does not create duplicate entries
tokens = word_tokenize(novel_text.lower())

print(f'Total tokens (words + punctuation): {len(tokens):,}')
print()
print('Sample tokens:', tokens[:30])

## 3. Filter — remove stopwords and punctuation

**Stopwords** are high-frequency words that carry little meaning on their own: `the`, `a`, `is`, `of`, `and`, …

If we kept them, they would dominate the frequency chart and obscure anything meaningful about Melville's vocabulary.

We also discard any token that is not alphabetic — removing punctuation marks, numbers, and artefacts from the encoding.

In [ ]:
from nltk.corpus import stopwords

english_stopwords = set(stopwords.words('english'))

# Keep a token only if:
#   1. It is composed entirely of alphabetic characters (no punctuation, no digits)
#   2. It is not a stopword
filtered_tokens = [
    token for token in tokens
    if token.isalpha() and token not in english_stopwords
]

print(f'Tokens after filtering: {len(filtered_tokens):,}')
print(f'Removed {len(tokens) - len(filtered_tokens):,} stopwords and punctuation tokens.')
print()
print('Sample filtered tokens:', filtered_tokens[:20])

## 4. Frequency distribution

`nltk.FreqDist` builds a mapping of `word → count` over the entire token list. It behaves like a Python `Counter` but with additional NLP-specific methods.

We use it to extract the top-N most frequent words and then visualise them.

In [ ]:
from nltk import FreqDist

freq_dist = FreqDist(filtered_tokens)

print(f'Unique words (vocabulary size): {len(freq_dist):,}')
print()
print('Top 10 words:')
for rank, (word, count) in enumerate(freq_dist.most_common(10), start=1):
    print(f'  {rank:>2}. {word:<20} {count:>5} occurrences')

## 5. Visualise — top-N word frequency bar chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

TOP_N = 25  # Change this to show more or fewer words

top_words, top_counts = zip(*freq_dist.most_common(TOP_N))

fig, ax = plt.subplots(figsize=(14, 6))

bars = ax.bar(top_words, top_counts, color='steelblue', edgecolor='white', linewidth=0.5)

# Annotate each bar with its count
for bar, count in zip(bars, top_counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 8,
        str(count),
        ha='center', va='bottom', fontsize=8, color='#444'
    )

ax.set_title(f'Top {TOP_N} Most Frequent Words in Moby Dick\n(stopwords removed)', fontsize=14, pad=16)
ax.set_xlabel('Word', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.tick_params(axis='x', rotation=40)
ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor('#f9f9f9')
fig.tight_layout()

plt.savefig('top_words.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to top_words.png')

## 6. Verify Zipf's Law

Zipf's Law predicts that if you plot **log(rank)** on the x-axis against **log(frequency)** on the y-axis, the points should fall along a straight line with a slope of approximately **−1**.

This is the power-law relationship: `frequency ∝ 1 / rank`.

We plot the empirical data from Moby Dick alongside a theoretical Zipf line to see how closely the novel conforms.

In [ ]:
import numpy as np

# Extract all words sorted by frequency (FreqDist.most_common returns them in descending order)
all_words, all_counts = zip(*freq_dist.most_common())

ranks      = np.arange(1, len(all_counts) + 1)   # 1, 2, 3, ..., vocab_size
counts     = np.array(all_counts, dtype=float)

# Theoretical Zipf: frequency of rank r = C / r, where C = frequency of rank 1
zipf_line  = counts[0] / ranks

fig, ax = plt.subplots(figsize=(10, 6))

ax.loglog(ranks, counts,    color='steelblue', linewidth=1.2,   label='Moby Dick (empirical)')
ax.loglog(ranks, zipf_line, color='tomato',    linewidth=1.5,
          linestyle='--', label="Zipf's Law (slope = −1)")

# Annotate a handful of the most common words directly on the plot
ANNOTATE_TOP = 8
for i in range(ANNOTATE_TOP):
    ax.annotate(
        all_words[i],
        xy=(ranks[i], counts[i]),
        xytext=(ranks[i] * 1.6, counts[i] * 0.85),
        fontsize=8.5, color='#333',
        arrowprops=dict(arrowstyle='->', color='#aaa', lw=0.8)
    )

ax.set_title("Zipf's Law in Moby Dick\n(log-log scale)", fontsize=14, pad=16)
ax.set_xlabel('Rank (log scale)', fontsize=11)
ax.set_ylabel('Frequency (log scale)', fontsize=11)
ax.legend(fontsize=10)
ax.spines[['top', 'right']].set_visible(False)
ax.set_facecolor('#f9f9f9')
ax.grid(True, which='both', linestyle=':', linewidth=0.5, alpha=0.5)
fig.tight_layout()

plt.savefig('zipf_law.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to zipf_law.png')

## 7. Quick stats summary

In [ ]:
total_tokens  = len(filtered_tokens)
vocab_size    = len(freq_dist)
top_word, top_count = freq_dist.most_common(1)[0]

# Type-token ratio: how lexically rich is the text?
# Higher = more diverse vocabulary.
ttr = vocab_size / total_tokens

print('=== Corpus summary ===')
print(f'  Total meaningful tokens : {total_tokens:>8,}')
print(f'  Unique words (vocab)    : {vocab_size:>8,}')
print(f'  Type-token ratio (TTR)  : {ttr:>8.4f}')
print(f'  Most common word        : "{top_word}" ({top_count:,} times)')
print()
print('Words that appear only once (hapax legomena):', freq_dist.hapaxes()[:10], '...')

---

## What to build next

This notebook gives you the core pipeline. Here is a natural progression to extend it:

| Extension | What it adds |
|---|---|
| **Swap the book** | Change `GUTENBERG_URL` to any other Project Gutenberg plain-text URL and re-run |
| **Sentiment arc** | Split by chapter, score each with `vaderSentiment`, plot the emotional trajectory |
| **Bigrams / trigrams** | Use `nltk.bigrams()` to find the most common two-word phrases instead of single words |
| **TF-IDF across multiple books** | Compare which words are *uniquely* characteristic of each author, not just frequent |
| **Author fingerprinting** | Extract stylistic features (sentence length, vocabulary richness) and train a classifier |

Each row is a natural next project that builds directly on the skills introduced here.